# Data Collection and Validation

**Inflation-Aware Portfolio Optimization**

This notebook loads and validates the frozen ETF and Canadian inflation datasets used in the research.

The main analysis uses frozen data to preserve historical reproducibility.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Run this notebook from the repository root.
PROJECT_ROOT = Path.cwd()

assert (PROJECT_ROOT / 'data/processed').is_dir(), (
    'Run the notebook from the repository root.'
)

DATA_DIR = PROJECT_ROOT / 'data/processed'
print('Project root:', PROJECT_ROOT)

## Load Frozen Research Data

The saved datasets are used instead of refreshing historical market prices.

In [ ]:
def load_csv(filename):
    return pd.read_csv(
        DATA_DIR / filename,
        index_col=0,
        parse_dates=True
    )

core = load_csv('core_returns.csv')
expanded = load_csv('expanded_returns.csv')
prices = load_csv('expanded_monthly_prices.csv')
inflation = load_csv('canada_monthly_inflation.csv')

for name, df in {
    'Core': core,
    'Expanded': expanded,
    'Prices': prices,
    'Inflation': inflation
}.items():
    print(name, df.shape)

## Data Validation

Check dimensions, missing values, asset consistency, and return reconstruction.

In [ ]:
assert core.shape == (161, 5)
assert expanded.shape == (161, 7)
assert prices.shape == (162, 7)
assert inflation.shape == (99, 1)

for df in [core, expanded, prices, inflation]:
    assert not df.isna().any().any()

assert core.index.equals(expanded.index)
assert np.allclose(
    core.to_numpy(),
    expanded[core.columns].to_numpy(),
    rtol=0, atol=1e-12
)

reconstructed = prices.pct_change(
    fill_method=None
).loc[expanded.index, expanded.columns]

max_error = np.max(np.abs(
    reconstructed.to_numpy() - expanded.to_numpy()
))

assert max_error < 1e-12
print('Frozen data validation PASSED!')
print('Maximum error:', max_error)